# ALSApp - Plant disease classifier (on-device)

Trains a small MobileNetV2-based classifier on the public **PlantVillage** dataset
(38 classes across 14 crop species, ~54k leaf photos) and exports it as a `.tflite`
file small enough to ship inside the Flutter app and run fully offline on the phone.

**Before running:**
1. `Runtime` -> `Change runtime type` -> select a `T4 GPU`.
2. Get a free Kaggle API token: kaggle.com -> your avatar -> `Settings` -> `API` ->
   `Create New Token`. Open it and copy the `username` and `key` values.
3. In this Colab, click the key icon (🔑) in the left sidebar ("Secrets") and add two
   secrets: `KAGGLE_USERNAME` and `KAGGLE_KEY`, with notebook access enabled for both.
   (Same two secrets as the cattle notebook - reuse them if you already set those up.)

We originally used TensorFlow Datasets' bundled `plant_village`, but its
download source (Mendeley) now returns `403 Forbidden` for automated
downloads, so this pulls a Kaggle-hosted mirror of the same dataset instead.

**At the end** this notebook downloads two files - save them into
`frontend/assets/models/` in the app repo:
- `plant_disease.tflite`
- `plant_labels.txt`

In [ ]:
!pip install -q -U tensorflow scikit-learn kagglehub

In [ ]:
import os
from google.colab import userdata

os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
os.environ['KAGGLE_KEY'] = userdata.get('KAGGLE_KEY')

import kagglehub

dataset_path = kagglehub.dataset_download('abdallahalidev/plantvillage-dataset')
print('Downloaded to:', dataset_path)

In [ ]:
# Kaggle dataset folder layouts vary and aren't documented in a machine-
# readable way - print the tree so you can SEE the real structure. If the
# auto-detection in the next cell picks the wrong folder, set DATA_DIR there
# manually based on what you see printed here (prefer a "color" variant over
# "grayscale"/"segmented" if this mirror has those).
!find "{dataset_path}" -maxdepth 5 -type d | head -80

In [ ]:
import tensorflow as tf

def find_class_dir(root):
    """Prefer a 'color' subfolder if this mirror has color/grayscale/
    segmented variants - otherwise walk down while there's exactly one
    subdirectory, stopping at the first level with 2+ subdirectories
    (almost certainly the folder-per-class root)."""
    current = root
    for _ in range(6):
        subdirs = [d for d in os.scandir(current) if d.is_dir()]
        color = next((d for d in subdirs if d.name.lower() == 'color'), None)
        if color:
            current = color.path
            continue
        if len(subdirs) != 1:
            return current
        current = subdirs[0].path
    return current

DATA_DIR = find_class_dir(dataset_path)  # override manually here if needed
print('Using DATA_DIR =', DATA_DIR)
print('Detected class folders:', [d.name for d in os.scandir(DATA_DIR) if d.is_dir()])

In [ ]:
IMG_SIZE = 224
BATCH_SIZE = 32

train_ds = tf.keras.utils.image_dataset_from_directory(
    DATA_DIR, validation_split=0.1, subset='training', seed=42,
    image_size=(IMG_SIZE, IMG_SIZE), batch_size=BATCH_SIZE)
val_ds = tf.keras.utils.image_dataset_from_directory(
    DATA_DIR, validation_split=0.1, subset='validation', seed=42,
    image_size=(IMG_SIZE, IMG_SIZE), batch_size=BATCH_SIZE)

class_names = train_ds.class_names
num_classes = len(class_names)
print(num_classes, 'classes:', class_names)

In [ ]:
import numpy as np
from sklearn.metrics import classification_report

augment = tf.keras.Sequential([
    tf.keras.layers.RandomFlip('horizontal'),
    tf.keras.layers.RandomRotation(0.08),
    tf.keras.layers.RandomBrightness(0.15),
    tf.keras.layers.RandomContrast(0.15),
])

train_ds = train_ds.map(lambda x, y: (augment(x, training=True), y),
                         num_parallel_calls=tf.data.AUTOTUNE)
train_ds = train_ds.prefetch(tf.data.AUTOTUNE)
val_ds = val_ds.prefetch(tf.data.AUTOTUNE)

In [ ]:
# Rescaling is baked into the model itself (not done in the tf.data pipeline)
# so the Flutter side can feed plain 0-255 RGB pixels with no normalization math.
base_model = tf.keras.applications.MobileNetV2(
    input_shape=(IMG_SIZE, IMG_SIZE, 3), include_top=False, weights='imagenet')
base_model.trainable = False

inputs = tf.keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
x = tf.keras.layers.Rescaling(1. / 127.5, offset=-1)(inputs)
x = base_model(x, training=False)
x = tf.keras.layers.GlobalAveragePooling2D()(x)
x = tf.keras.layers.Dropout(0.2)(x)
outputs = tf.keras.layers.Dense(num_classes, activation='softmax')(x)
model = tf.keras.Model(inputs, outputs)

model.compile(optimizer=tf.keras.optimizers.Adam(1e-3),
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])
model.summary()

In [ ]:
# Phase 1: train just the new head, backbone frozen.
model.fit(train_ds, validation_data=val_ds, epochs=6)

In [ ]:
# Phase 2: unfreeze the top of the backbone and fine-tune a few epochs at a
# low learning rate - usually a solid accuracy bump for not much extra time.
base_model.trainable = True
for layer in base_model.layers[:-30]:
    layer.trainable = False

model.compile(optimizer=tf.keras.optimizers.Adam(1e-5),
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])
model.fit(train_ds, validation_data=val_ds, epochs=4)

In [ ]:
# Honest held-out numbers before we ship this anywhere.
y_true, y_pred = [], []
for images, labels in val_ds:
    preds = model.predict(images, verbose=0)
    y_true.extend(labels.numpy())
    y_pred.extend(np.argmax(preds, axis=1))

print(classification_report(y_true, y_pred, target_names=class_names, zero_division=0))

In [ ]:
converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]  # dynamic-range quantization, smaller file
tflite_model = converter.convert()

with open('plant_disease.tflite', 'wb') as f:
    f.write(tflite_model)
with open('plant_labels.txt', 'w') as f:
    f.write('\n'.join(class_names))

import os
print(f"plant_disease.tflite: {os.path.getsize('plant_disease.tflite') / 1e6:.1f} MB")

In [ ]:
from google.colab import files
files.download('plant_disease.tflite')
files.download('plant_labels.txt')